<a href="https://colab.research.google.com/github/blmvv/MachineLearning/blob/Machine-Learning/%D0%9B%D0%A0_9_%D0%A0%D0%B5%D0%BA%D0%BE%D0%BC%D0%B5%D0%BD%D0%B4%D0%B0%D1%86%D1%96%D0%B9%D0%BD%D1%96_%D1%81%D0%B8%D1%81%D1%82%D0%B5%D0%BC%D0%B8_%D0%91%D1%96%D0%BB%D0%B8%D0%BC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**ПІБ студента:** Білим Вадим Юрійович  
**Група:** ФІТ 4-10  
**Дата:** 22.11.2026

> **Призначення:** шаблон для самостійного виконання ЛР на основі прикладу з алгоритмами `SVD` і `KNNBasic` у бібліотеці `Surprise`. Комірки з позначкою **TODO** потрібно заповнити власним кодом. Це не готовий розв’язок і не заміна офіційної умови роботи.

**Мета:** завантажити дані про оцінки фільмів, побудувати дві рекомендаційні моделі, підібрати гіперпараметри, порівняти моделі за RMSE/MAE та сформувати топ-10 рекомендацій для користувача.

**Дані:** вбудований набір `MovieLens 100K` (`ml-100k`) із бібліотеки `Surprise`.

**Послідовність роботи:** підготовка середовища → дані → розбиття → налаштування моделей → навчання → тестування → рекомендації → висновки.

## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [ ]:
%pip install -q \
    numpy==2.2.6 \
    scipy==1.15.3 \
    scikit-learn==1.6.1 \
    pandas==2.2.3 \
    scikit-surprise==1.1.5

In [ ]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise

from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split as sklearn_train_test_split

print('Python:', sys.version.split()[0])
print('NumPy:', np.__version__)
print('SciPy:', scipy.__version__)
print('scikit-learn:', sklearn.__version__)
print('pandas:', pd.__version__)
print('Surprise:', surprise.__version__)

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.16.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [ ]:
# Завантаження даних (готова частина шаблону)
data = Dataset.load_builtin('ml-100k', prompt=False)

ratings_df = pd.DataFrame(
    data.raw_ratings,
    columns=['user_id', 'item_id', 'rating', 'timestamp']
)

print('Розмір:', ratings_df.shape)
display(ratings_df.head())

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмір: (100000, 4)


,user_id,item_id,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


In [ ]:
print("Кількість користувачів:", ratings_df['user_id'].nunique())
print("Кількість фільмів:", ratings_df['item_id'].nunique())

print("Мінімальна оцінка:", ratings_df['rating'].min())
print("Максимальна оцінка:", ratings_df['rating'].max())
print("Середня оцінка:", ratings_df['rating'].mean())

print("Пропущені значення:")
print(ratings_df.isnull().sum())

print("Розподіл оцінок:")
print(ratings_df['rating'].value_counts().sort_index())

Кількість користувачів: 943
Кількість фільмів: 1682
Мінімальна оцінка: 1.0
Максимальна оцінка: 5.0
Середня оцінка: 3.52986
Пропущені значення:
user_id      0
item_id      0
rating       0
timestamp    0
dtype: int64
Розподіл оцінок:
rating
1.0     6110
2.0    11370
3.0    27145
4.0    34174
5.0    21201
Name: count, dtype: int64


**Короткі спостереження щодо даних:**

Набір даних містить 100 000 оцінок від 943 користувачів для 1682 фільмів. Оцінки знаходяться в діапазоні від 1 до 5. Ці дані дозволяють дослідити вподобання користувачів та створити систему рекомендацій.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [ ]:
train_df, test_df = sklearn_train_test_split(
    ratings_df,
    test_size=0.2,
    random_state=42)

print("Навчальна вибірка:", train_df.shape)
print("Тестова вибірка:", test_df.shape)

Навчальна вибірка: (80000, 4)
Тестова вибірка: (20000, 4)


In [ ]:
reader = Reader(rating_scale=data.reader.rating_scale)

train_data = Dataset.load_from_df(
    train_df[['user_id', 'item_id', 'rating']],
    reader)

testset = list(
    test_df[['user_id', 'item_id', 'rating']]
    .itertuples(index=False, name=None))

print("Навчальних оцінок:", len(train_df))
print("Тестових оцінок:", len(testset))

Навчальних оцінок: 80000
Тестових оцінок: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [ ]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]}
}

**Обґрунтування вибору параметрів:**

Для SVD перевіряємо різну кількість прихованих факторів, швидкість навчання та регуляризацію. Для KNNBasic змінюємо кількість сусідів і спосіб визначення схожості. Ці параметри допомагають знайти налаштування, за яких моделі найточніше прогнозують оцінки.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [ ]:
grid_search_svd = GridSearchCV(
    SVD,
    param_grid_svd,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1)

grid_search_svd.fit(train_data)

In [ ]:
grid_search_knn = GridSearchCV(
    KNNBasic,
    param_grid_knn,
    measures=['rmse', 'mae'],
    cv=3,
    n_jobs=-1)

grid_search_knn.fit(train_data)

In [ ]:
print("SVD:")
print("RMSE:", grid_search_svd.best_score['rmse'])
print("MAE:", grid_search_svd.best_score['mae'])
print("Параметри:", grid_search_svd.best_params['rmse'])

print("\nKNNBasic:")
print("RMSE:", grid_search_knn.best_score['rmse'])
print("MAE:", grid_search_knn.best_score['mae'])
print("Параметри:", grid_search_knn.best_params['rmse'])

SVD:
RMSE: 0.9433629386704921
MAE: 0.7441770302555929
Параметри: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

KNNBasic:
RMSE: 1.0263712257013335
MAE: 0.8140105976221731
Параметри: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**

| Модель | Найкращий CV RMSE | CV MAE | Найкращі параметри за RMSE |
|:--|--:|--:|:--|
| SVD | 0.9433629386704921 | 0.7441770302555929 | {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05} |
| KNNBasic | 1.0263712257013335 | 0.8140105976221731 | {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}} |

За результатами кросвалідації модель SVD показала кращу точність, ніж KNNBasic. Її RMSE становить 0.9434, а MAE — 0.7442. Для KNNBasic ці значення вищі: 1.0264 та 0.8140. Найкращими параметрами SVD стали 50 факторів, швидкість навчання 0.01 та регуляризація 0.05. Для KNNBasic — 40 сусідів із використанням косинусної схожості між користувачами.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [ ]:
best_svd_params = grid_search_svd.best_params['rmse']
best_knn_params = grid_search_knn.best_params['rmse']

trainset = train_data.build_full_trainset()

best_svd = SVD(
    **best_svd_params,
    random_state=42)

best_knn = KNNBasic(**best_knn_params)

best_svd.fit(trainset)
best_knn.fit(trainset)

print("Обидві моделі навчено")

Computing the cosine similarity matrix...
Done computing similarity matrix.
Обидві моделі навчено


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [ ]:
svd_predictions = best_svd.test(testset)
knn_predictions = best_knn.test(testset)

svd_rmse = accuracy.rmse(svd_predictions, verbose=False)
svd_mae = accuracy.mae(svd_predictions, verbose=False)

knn_rmse = accuracy.rmse(knn_predictions, verbose=False)
knn_mae = accuracy.mae(knn_predictions, verbose=False)

print("SVD:")
print("RMSE:", svd_rmse)
print("MAE:", svd_mae)

print("\nKNNBasic:")
print("RMSE:", knn_rmse)
print("MAE:", knn_mae)

SVD:
RMSE: 0.9191746414249822
MAE: 0.7233554064545319

KNNBasic:
RMSE: 1.0119527444069378
MAE: 0.7984207578945732


In [ ]:
results = pd.DataFrame({
    'Model': ['SVD', 'KNNBasic'],
    'RMSE': [svd_rmse, knn_rmse],
    'MAE': [svd_mae, knn_mae]})

results = results.sort_values('RMSE')

display(results)

,Model,RMSE,MAE
0,SVD,0.919175,0.723355
1,KNNBasic,1.011953,0.798421


**Таблиця тестових результатів:**

| Модель | Test RMSE | Test MAE |
|:--|--:|--:|
| SVD | 0.919175 | 0.723355 |
| KNNBasic | 1.011953 | 0.798421 |

**Порівняння з CV:** На тестових даних модель SVD також показала кращі результати: RMSE — 0.9192, MAE — 0.7234. У KNNBasic ці показники становлять 1.0120 та 0.7984. Результати тестування незначно кращі за результати кросвалідації, тому можна зробити висновок, що моделі працюють досить стабільно.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [ ]:
if grid_search_svd.best_score['rmse'] < grid_search_knn.best_score['rmse']:
    best_model = best_svd
    print("Обрана модель: SVD")
else:
    best_model = best_knn
    print("Обрана модель: KNNBasic")

user_id = '196'

try:
    inner_user_id = trainset.to_inner_uid(user_id)

    rated_items = {
        trainset.to_raw_iid(item_id)
        for item_id, rating in trainset.ur[inner_user_id]
    }

    all_items = {
        trainset.to_raw_iid(i)
        for i in trainset.all_items()
    }

    unrated_items = all_items - rated_items

    recommendations = []

    for item_id in unrated_items:
        predicted_rating = best_model.predict(
            uid=user_id,
            iid=item_id
        ).est

        recommendations.append((item_id, predicted_rating))

    recommendations.sort(key=lambda x: x[1], reverse=True)

    top_10 = pd.DataFrame(
        recommendations[:10],
        columns=['item_id', 'predicted_rating']
    )

    display(top_10)

except ValueError:
    print("Користувача немає у навчальній вибірці")

Обрана модель: SVD


,item_id,predicted_rating
0,320,4.490358
1,64,4.470680
2,480,4.413710
3,313,4.401920
4,513,4.399624
5,483,4.398092
6,318,4.392102
7,656,4.385379
8,22,4.377388
9,272,4.376010


## 9. Перевірка виконання

Позначте після реалізації:

- [+] Дані `ml-100k` завантажено та перевірено.
- [+] Є розбиття 80/20 з фіксованим `random_state`.
- [+] `Reader`, `train_data` і `testset` створені коректно.
- [+] GridSearchCV для **обох** моделей виконано лише на `train_data`.
- [+] Для SVD і KNNBasic показано параметри та результати CV.
- [+] Моделі навчено на повній навчальній частині.
- [+] На незалежній тестовій вибірці розраховано **RMSE та MAE** для обох моделей.
- [+] Наведено порівняльну таблицю та інтерпретацію.
- [+] Сформовано топ-10 рекомендацій без уже оцінених фільмів.
- [+] Усі комірки виконуються послідовно після перезапуску середовища.

## 10. Висновки

Заповніть **власними результатами**:

1. Яку модель обрано за кросвалідацією і чому?
2. Які найкращі гіперпараметри отримано для обох моделей?
3. Які підсумкові значення RMSE та MAE на тестовій вибірці?
4. Чи є різниця між результатами CV та тестування?
5. Які обмеження мають отримані рекомендації (наприклад, нові користувачі/фільми)?

**Висновок:** У ході лабораторної роботи було створено та досліджено рекомендаційні системи SVD і KNNBasic на наборі даних MovieLens 100K. Для обох моделей підібрано найкращі параметри за допомогою кросвалідації. Кращою виявилася модель SVD з параметрами: 50 факторів, швидкість навчання 0.01 та регуляризація 0.05. На тестових даних вона отримала RMSE 0.9192 та MAE 0.7234, що краще за KNNBasic з 40 сусідами та косинусною схожістю. Результати тестування мало відрізняються від кросвалідації. За допомогою SVD було реалізовано топ-10 рекомендацій фільмів для користувача. Основним обмеженням системи є складність створення точних рекомендацій для нових користувачів і фільмів без оцінок.